# Notebook 03 — Covarianzas alternativas para el GP

Comparamos cuatro funciones de covarianza para el componente $b(s)$ de GPBoost:
exponential, Matérn $\nu=1.5$, Matérn $\nu=2.5$ y gaussiana. La hipótesis es que a mayor *smoothness* de la covarianza, peor desempeño en datos con discontinuidades como los simulados.

## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Colab y, en ese caso, instala `gpboost` y `shap` y clona el repositorio para tener acceso a los datos derivados (`data/`) y a los modelos entrenados (`modelos/`). En Jupyter local no hace nada.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab — instalando dependencias...')
    !pip -q install gpboost shap
    if not os.path.isdir('/content/repo'):
        !git clone -q https://github.com/linaq11/examen-2-gpboost.git /content/repo
    os.chdir('/content/repo')
    print('CWD:', os.getcwd())
else:
    # En local: asume que el notebook está en repo/notebooks/
    here = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    if os.path.basename(here) == 'notebooks':
        os.chdir(os.path.dirname(here))
    print('CWD:', os.getcwd())

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gpboost as gpb

np.random.seed(1)
DATA = 'data'
MOD  = 'modelos'
FIG  = 'figuras'
os.makedirs(FIG, exist_ok=True)

## 1. Datos

In [ ]:
df = pd.read_csv(os.path.join(DATA, 'simulado_train_seed1.csv'))
X      = df[['X1','X2']].values
y      = df['y'].values
coords = df[['s_1','s_2']].values
tabla = pd.read_csv(os.path.join(DATA, 'tabla_covarianzas.csv'))
tabla

## 2. Resultados

RMSE sobre el test grid 50×50 ordenado por suavidad creciente de la covarianza:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#2E86AB','#5DADE2','#F39C12','#E74C3C']
ax.bar(tabla['cov_function'], tabla['RMSE'], color=colors)
for i, v in enumerate(tabla['RMSE']):
    ax.text(i, v+0.005, f'{v:.4f}', ha='center')
ax.set_ylabel('RMSE test grid 50×50')
ax.set_title('Comparativa de funciones de covarianza')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig6_covarianzas_RMSE.png'), dpi=120, bbox_inches='tight')
plt.show()

### Lectura

El RMSE crece de forma monótona con la suavidad de la covarianza. La exponencial ($\nu=0.5$, no diferenciable) gana, seguida por Matérn $\nu=1.5$, Matérn $\nu=2.5$ y al final la gaussiana (infinitamente suave). Esto es consistente con la naturaleza no suave de la simulación, que incluye una discontinuidad en $x_1=0.5$. El criterio general: **a más rugosidad en los datos, covarianza menos suave**. La elección por defecto del artículo (exponencial) es la correcta para este caso.

## 3. Sintaxis de GPBoost para Matérn

Recordatorio para el reporte: GPBoost expone Matérn con dos argumentos:

In [ ]:
# Sintaxis correcta para Matérn (nu se pasa como cov_fct_shape, no en el nombre)
# Ejemplo (no ejecutamos aquí, solo documentación):
# gp_matern_15 = gpb.GPModel(gp_coords=coords, cov_function='matern', cov_fct_shape=1.5)
# gp_matern_25 = gpb.GPModel(gp_coords=coords, cov_function='matern', cov_fct_shape=2.5)
print('Sintaxis documentada en cov_function="matern", cov_fct_shape=ν')